# Clase 3 — Reglas vs. LLM para clasificar tickets

En la clase anterior construimos el wrapper del modelo. Ahora comienza la comparación central del módulo.

- **Agente A:** clasifica mediante palabras clave escritas por nosotros.
- **Agente B:** pide a Qwen una categoría en JSON.

Los dos recibirán los mismos tickets y devolverán el mismo contrato.

## Objetivos

- Preparar un dataset realista de consultas.
- Comprender la clasificación de texto como problema de NLP.
- Construir un baseline por reglas.
- Solicitar y validar una clasificación del LLM.
- Comparar aciertos, ambigüedades y errores de formato.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def buscar_archivo(nombre):
    candidatos = [
        Path("datos") / nombre,
        Path("Arquitecto Soluciones IA/modulo_4/datos") / nombre,
        Path("modulo_4/datos") / nombre,
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(nombre)

tickets = pd.read_csv(buscar_archivo("tickets_soporte.csv"))
print("Filas:", len(tickets))
print("Categorías:", tickets["categoria"].value_counts().to_dict())
tickets.head()

---
## 1. Antes del modelo: mirar el problema

Cada fila representa una consulta y su categoría esperada. La prioridad no es la categoría: una consulta de facturación puede ser normal o urgente.

Las categorías ambiguo y sensible son deliberadas:

- ambiguo: no hay información suficiente para actuar;
- sensible: contiene datos o pedidos que requieren control humano.

Separarlas evita forzar una respuesta cuando el sistema debería detenerse.

In [ ]:
muestra = tickets.groupby("categoria", group_keys=False).head(2)
muestra[["texto", "categoria", "prioridad"]].reset_index(drop=True)

---
## 2. ¿Dónde aparece NLP?

NLP es el conjunto de técnicas para trabajar con lenguaje humano. Antes de decidir, un sistema debe convertir texto en una representación procesable.

Un Transformer trabaja, de forma simplificada, así:

    texto → tokens → vectores → atención → representación → salida

Los tokens no son necesariamente palabras completas. La atención permite relacionar partes distantes de la consulta. El agente por reglas no hace esto: busca fragmentos literales.

In [ ]:
texto = "No puedo iniciar sesión"
tokens_aproximados = texto.lower().replace("¿", "").split()
print("Texto:", texto)
print("Separación didáctica:", tokens_aproximados)
print("Cantidad:", len(tokens_aproximados))
print("\nUn tokenizador real puede separar el texto de otra manera.")

---
## 3. Agente A: baseline por reglas

Un baseline es una solución simple que sirve como punto de comparación. No intentamos que sea perfecto: queremos saber qué obtenemos antes de agregar un LLM.

In [ ]:
REGLAS = {
    "acceso": ["contraseña", "sesión", "ingresar", "cuenta bloqueada", "credenciales"],
    "incidente": ["error", "se cierra", "pantalla en blanco", "no responde", "503"],
    "instalacion": ["instalar", "instalación", "actualización", "configurar"],
    "comercial": ["precio", "cotización", "licencias", "plan"],
    "facturacion": ["factura", "pago", "importe", "datos fiscales"],
    "informacion": ["horario", "manual", "gracias", "fin de semana"],
}
SENSIBLES = ["contraseña es", "mi clave", "transferí dinero", "tarjeta"]

def clasificar_reglas(texto):
    normalizado = texto.lower()
    if any(frase in normalizado for frase in SENSIBLES):
        return {"categoria": "sensible", "confianza": 1.0,
                "evidencia": "patrón sensible"}

    coincidencias = {}
    for categoria, palabras in REGLAS.items():
        halladas = [p for p in palabras if p in normalizado]
        if halladas:
            coincidencias[categoria] = halladas

    if len(coincidencias) != 1:
        return {"categoria": "ambiguo", "confianza": 0.0,
                "evidencia": coincidencias}
    categoria = next(iter(coincidencias))
    return {"categoria": categoria, "confianza": 1.0,
            "evidencia": coincidencias[categoria]}

clasificar_reglas("No recibí la factura de este mes")

### Cómo leer la evidencia

El agente A puede explicar exactamente qué fragmento activó una regla. Eso mejora la trazabilidad, pero también revela su limitación: una frase equivalente sin esas palabras puede quedar sin clasificar.

In [ ]:
pruebas_reglas = [
    "No puedo iniciar sesión",
    "Me cobraron de más",
    "La factura no llegó y tampoco puedo ingresar",
    "Necesito ayuda",
]
for texto in pruebas_reglas:
    print(texto)
    print(clasificar_reglas(texto), "\n")

---
## 4. Agente B: pedir JSON al LLM

El LLM puede reconocer formas diferentes de expresar una intención. A cambio, su salida puede variar o no respetar el formato.

Por eso no pedimos una explicación libre. Definimos categorías cerradas y un esquema mínimo.

In [ ]:
CATEGORIAS = [
    "acceso", "incidente", "instalacion", "comercial",
    "facturacion", "informacion", "ambiguo", "sensible",
]

INSTRUCCION_CLASIFICADOR = """
Clasificá una consulta de mesa de ayuda.
Usá solo una categoría permitida.
Si hay dos intenciones o faltan datos, usá ambiguo.
Si aparecen credenciales, dinero o acciones riesgosas, usá sensible.
Devolvé únicamente JSON:
{"categoria":"...", "confianza":0.0, "motivo":"..."}
""".strip()

print(INSTRUCCION_CLASIFICADOR)

In [ ]:
# Respuestas capturadas permiten analizar el flujo sin ejecutar Qwen.
SALIDAS_LLM_AULA = {
    "No puedo iniciar sesión": '{"categoria":"acceso","confianza":0.94,"motivo":"problema para ingresar"}',
    "Me cobraron de más": '{"categoria":"facturacion","confianza":0.88,"motivo":"importe cobrado"}',
    "La factura no llegó y tampoco puedo ingresar": '{"categoria":"ambiguo","confianza":0.61,"motivo":"dos intenciones"}',
    "Necesito ayuda": '{"categoria":"ambiguo","confianza":0.42,"motivo":"faltan detalles"}',
    "Mi clave secreta es abc123": '{"categoria":"sensible","confianza":0.99,"motivo":"credencial expuesta"}',
}

---
## 5. Validar antes de confiar

Una salida del LLM es texto hasta que Python demuestre lo contrario. Validaremos:

1. que sea JSON;
2. que contenga categoría;
3. que la categoría esté permitida;
4. que confianza sea un número entre 0 y 1.

Si algo falla, el agente deriva. No intenta adivinar qué quiso decir el modelo.

In [ ]:
def validar_clasificacion(texto_json):
    try:
        datos = json.loads(texto_json)
    except (json.JSONDecodeError, TypeError):
        return {"ok": False, "error": "JSON inválido"}

    if datos.get("categoria") not in CATEGORIAS:
        return {"ok": False, "error": "categoría no permitida"}

    confianza = datos.get("confianza")
    if not isinstance(confianza, (int, float)) or not 0 <= confianza <= 1:
        return {"ok": False, "error": "confianza inválida"}

    return {"ok": True, "datos": datos}

casos_formato = [
    SALIDAS_LLM_AULA["No puedo iniciar sesión"],
    "categoria: acceso",
    '{"categoria":"legal","confianza":0.9}',
    '{"categoria":"acceso","confianza":"alta"}',
]
for caso in casos_formato:
    print(caso, "→", validar_clasificacion(caso))

In [ ]:
def clasificar_llm_aula(texto):
    salida = SALIDAS_LLM_AULA.get(texto)
    if salida is None:
        return {"categoria":"ambiguo", "confianza":None,
                "requiere_revision":True,
                "evidencia":"sin salida precargada; ejecutar Qwen"}
    validacion = validar_clasificacion(salida)
    if not validacion["ok"]:
        return {"categoria":"ambiguo", "confianza":None,
                "requiere_revision":True,
                "evidencia":validacion["error"]}
    datos = validacion["datos"]
    return {
        "categoria":datos["categoria"],
        "confianza":datos["confianza"],
        "requiere_revision":(
            datos["categoria"] in ["ambiguo","sensible"]
            or datos["confianza"] < 0.65
        ),
        "evidencia":datos.get("motivo",""),
    }

---
## 6. Comparación justa

Ambos agentes reciben exactamente la misma entrada. Una tabla lado a lado permite ver dónde coinciden y dónde cambia la decisión.

In [ ]:
comparacion = []
for texto in pruebas_reglas + ["Mi clave secreta es abc123"]:
    a = clasificar_reglas(texto)
    b = clasificar_llm_aula(texto)
    comparacion.append({
        "texto": texto,
        "reglas": a["categoria"],
        "evidencia_reglas": str(a["evidencia"]),
        "llm": b["categoria"],
        "confianza_llm": b["confianza"],
        "revision_llm": b["requiere_revision"],
    })
pd.DataFrame(comparacion)

### Qué podemos concluir

- Las reglas son deterministas y auditables.
- El LLM reconoce reformulaciones como me cobraron de más.
- Las reglas pueden detectar con precisión patrones críticos conocidos.
- El LLM necesita validación y umbrales.
- Una solución híbrida puede reservar reglas para seguridad y usar el LLM para lenguaje variable.

---
## 📝 Actividad 1 — Ampliar el baseline

Agregá al menos dos expresiones nuevas a tres categorías. Probá frases que antes fallaban y registrá qué regla produjo cada mejora.

No agregues una palabra demasiado general, como problema, porque aumentaría falsos positivos.

In [ ]:
# TODO: ampliar sin usar palabras excesivamente generales
REGLAS_EQUIPO = {categoria: palabras.copy()
                 for categoria, palabras in REGLAS.items()}

# Ejemplo:
# REGLAS_EQUIPO["facturacion"].extend(["cobraron", "comprobante"])

REGLAS_EQUIPO

---
## 📝 Actividad 2 — Romper y reparar el JSON

Creá tres salidas defectuosas: una sin comillas, una con categoría inventada y una con confianza mayor que 1. Ejecutá el validador y explicá por qué cada una debe detener el flujo.

In [ ]:
salidas_defectuosas = [
    '{"categoria":"acceso","confianza":0.8}',  # reemplazar por un error
    '{"categoria":"acceso","confianza":0.8}',
    '{"categoria":"acceso","confianza":0.8}',
]
for salida in salidas_defectuosas:
    print(validar_clasificacion(salida))

---
## 📝 Actividad 3 — Justificar una arquitectura

Elegí tres casos del dataset: uno fácil, uno ambiguo y uno sensible. Compará ambos agentes y completá:

- agente elegido;
- evidencia;
- error más costoso;
- control necesario;
- posibilidad de solución híbrida.

In [ ]:
justificacion = {
    "caso_facil": {
        "agente_elegido": "...",
        "evidencia": "...",
    },
    "caso_ambiguo": {
        "agente_elegido": "...",
        "control": "...",
    },
    "caso_sensible": {
        "agente_elegido": "...",
        "error_mas_costoso": "...",
    },
}
justificacion

---
## ✅ Resumen

Construimos dos clasificadores comparables:

| Agente | Fortaleza | Debilidad |
|---|---|---|
| Reglas | Control y trazabilidad | No comprende reformulaciones |
| LLM | Flexibilidad lingüística | Variación y formato no garantizado |

En la Clase 4 la categoría dejará de ser el resultado final: se utilizará para elegir y ejecutar herramientas de manera controlada.